In [8]:
import pandas as pd

DATA_PATH = "/Users/sanjana/Desktop/Hype-Predictor/Data"

df_main = pd.read_csv(f"{DATA_PATH}/clean_data.csv")
trends = pd.read_csv(f"{DATA_PATH}/google_trends.csv")

trends.head()

,date,PS5 Pro,iPhone 17,Air Jordan 11,Owala FreeSip,Nvidia RTX 5090
0,2021-04-11,2,0,30,0,0
1,2021-04-18,2,0,47,0,0
2,2021-04-25,2,0,39,0,0
3,2021-05-02,2,0,61,0,0
4,2021-05-09,2,0,40,0,0


In [10]:
#reshaping the data
trends_long = trends.melt(
    id_vars=["date"],
    var_name="product",
    value_name="trend_interest"
)

trends_long["date"] = pd.to_datetime(trends_long["date"])

trends_long.head()

,date,product,trend_interest
0,2021-04-11,PS5 Pro,2
1,2021-04-18,PS5 Pro,2
2,2021-04-25,PS5 Pro,2
3,2021-05-02,PS5 Pro,2
4,2021-05-09,PS5 Pro,2


In [12]:
#Compute REAL Momentum (per product over time)
trends_long = trends_long.sort_values(["product", "date"])

# Previous value
trends_long["lag"] = trends_long.groupby("product")["trend_interest"].shift(1)

# Momentum = change over time
trends_long["momentum"] = trends_long["trend_interest"] - trends_long["lag"]

In [14]:
#ggregate Momentum per Product-We convert time-series → single feature per product:
momentum_df = trends_long.groupby("product").agg({
    "momentum": "mean"   # average growth
}).reset_index()

momentum_df.head()

,product,momentum
0,Air Jordan 11,0.022989
1,Nvidia RTX 5090,0.103448
2,Owala FreeSip,0.298851
3,PS5 Pro,0.057471
4,iPhone 17,0.076628


In [16]:
#scale it now 
min_m = momentum_df["momentum"].min()
max_m = momentum_df["momentum"].max()

momentum_df["momentum_score"] = 100 * (
    (momentum_df["momentum"] - min_m) / (max_m - min_m)
)

momentum_df.sort_values("momentum_score", ascending=False)

,product,momentum,momentum_score
2,Owala FreeSip,0.298851,100.000000
1,Nvidia RTX 5090,0.103448,29.166667
4,iPhone 17,0.076628,19.444444
3,PS5 Pro,0.057471,12.500000
0,Air Jordan 11,0.022989,0.000000


In [18]:
#merge with my dataset
df = df_main.merge(
    momentum_df[["product", "momentum_score"]],
    on="product"
)

In [20]:
def trend_label(x):
    if x > 70:
        return "🔥 Trending Up"
    elif x < 30:
        return "📉 Losing Hype"
    else:
        return "➡️ Stable"

df["trend_label"] = df["momentum_score"].apply(trend_label)

In [22]:
df.to_csv(f"{DATA_PATH}/enhanced_data.csv", index=False)

In [24]:
print("""
Owala FreeSip shows the highest momentum, indicating a rapid increase in search interest.
This is likely driven by social media virality and influencer engagement.

In contrast, products like Air Jordan 11 show declining momentum, suggesting post-release hype saturation.

Tech products like Nvidia RTX 5090 and iPhone 17 exhibit slower momentum, reflecting longer product cycles and gradual buildup of interest.
""")


Owala FreeSip shows the highest momentum, indicating a rapid increase in search interest.
This is likely driven by social media virality and influencer engagement.

In contrast, products like Air Jordan 11 show declining momentum, suggesting post-release hype saturation.

Tech products like Nvidia RTX 5090 and iPhone 17 exhibit slower momentum, reflecting longer product cycles and gradual buildup of interest.

